In [1]:
import os
from  dotenv import load_dotenv
load_dotenv()

groq_key=os.getenv("sample_qroq_api_key")

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("./attention.pdf")
doc=loader.load()
doc

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukas

In [3]:
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

/var/folders/hj/8fxfbdp516z1q9b5vhdr9yq00000gn/T/ipykernel_47740/780310583.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/Users/anurag2/Desktop/Udemy/Gen-AI/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)
final_doc=splitter.split_documents(doc)
final_doc

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to'),
 Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='reproduce the tables 

In [5]:
from langchain_chroma import Chroma

vector_db=Chroma.from_documents(embedding=embeddings,documents=final_doc)
vector_db

In [6]:
retriever=vector_db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x344cb3ed0>, search_kwargs={})

In [7]:
from langchain_groq import ChatGroq
import httpx


custom_http_client = httpx.Client(verify=False)
llm=ChatGroq(model="gemma2-9b-it",groq_api_key=groq_key,http_client=custom_http_client)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a8010b10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x341ce0b10>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x344c62750>)

In [8]:
# if "input" not in prompt.input_variables:
#         msg = (
#             "Expected `input` to be a prompt variable, "
#             f"but got {prompt.input_variables}"
#         )
#         raise ValueError(msg)

#     retrieve_documents: RetrieverOutputLike = RunnableBranch(
#         (
#             # Both empty string and empty list evaluate to False
#             lambda x: not x.get("chat_history", False),
#             # If no chat history, then we just pass input to retriever
#             (lambda x: x["input"]) | retriever,
#         ),
#         # If chat history, then we pass inputs to LLM chain, then to retriever
#         prompt | llm | StrOutputParser() | retriever,
#     ).with_config(run_name="chat_retriever_chain")
#     return retrieve_documents

In [9]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate

#create_history_aware_retriever create  a chain that takes conversation history and returns documents.
# If there is no chat_history, then the input is just passed directly to the retriever. 
# If there is chat_history, then the prompt and LLM will be used to generate a search query.
#  That search query is then passed to the retriever.

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

retriever_prompt=ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)

history_retriever=create_history_aware_retriever(llm=llm,retriever=retriever,prompt=retriever_prompt)
history_retriever


RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x344cb3ed0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk'

In [13]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

qa_prompt

ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.

In [15]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain


question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)
rag_chain=create_retrieval_chain(history_retriever,question_answer_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x344cb3ed0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated

Great question. Let's break it down clearly.

### TL;DR:

You pass `MessagesPlaceholder("chat_history")` to the `qa_prompt` **so that the system and the LLM have access to the ongoing conversation history**, which allows them to maintain context when answering the current user question.

---

### Let’s look at the full flow of your RAG chain:

1. **User asks a question** (possibly dependent on previous messages).
2. **`create_history_aware_retriever`**:

   * If there is `chat_history`, it uses the prompt (`retriever_prompt`) to turn the **current input + chat history** into a **standalone query**.
   * This helps the retriever fetch relevant documents even if the current user question refers to past context.
3. **Documents are retrieved**.
4. **`question_answer_chain`** (created via `create_stuff_documents_chain`) then takes:

   * the retrieved documents (`{context}`)
   * the **original chat history**
   * the **current question**
   * and uses these to generate an **answer** using `qa_prompt`.

---

### Why is `chat_history` needed in `qa_prompt`?

Imagine this conversation:

> **User (earlier):** What's the capital of France?
> **Assistant:** Paris.
> **User (now):** What's the population?

In this case:

* The retriever will first **reformulate** the question to:

  > "What is the population of Paris?"
* The retriever fetches documents about Paris.
* Now, when the LLM is answering with the `qa_prompt`, having the `chat_history` helps it **maintain continuity**, e.g., understand tone, follow-ups, or disambiguate pronouns.

So, we include `MessagesPlaceholder("chat_history")` in `qa_prompt` to:

* Give the LLM full context of the conversation.
* Allow it to produce coherent, contextual, and more helpful answers.
* Help it **not repeat information** already discussed, unless needed.
* Support **follow-up questions** effectively.

---

### Without `chat_history` in `qa_prompt`, you’d lose:

* Context awareness.
* Coherence in follow-up answers.
* Personalized tone and flow of the conversation.

---

### Summary

You need `MessagesPlaceholder("chat_history")` in `qa_prompt` because:

> It allows the LLM to consider the full conversation when generating the answer, not just the current question and retrieved documents.

This is crucial for building **intelligent, contextual RAG systems** that can handle **multi-turn conversations** effectively.


In [16]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

store={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [17]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain=RunnableWithMessageHistory(
    runnable=rag_chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

chain

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  chat_history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x3a4545b20>, input_messages_key='input', output_messages_key='answer', history_messages_key='chat_history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [18]:
config={
        "configurable": {"session_id": "abc123"}
    }
config


{'configurable': {'session_id': 'abc123'}}

In [19]:
chain.invoke(
    {
        "input":"what is encoder?"
    },
    config=config
)["answer"]

'The encoder is a component of a transformer model. \nIt consists of multiple identical layers, each containing two sublayers: a multi-head self-attention mechanism and a feed-forward network. \nThe encoder processes the input sequence and generates a representation of its meaning.  \n'

In [20]:
chain.invoke(
    {
        "input":"how many layers it has?"
    },
    config=config
)["answer"]

'The encoder has N = 6 layers.  \n'

In [21]:
store["abc123"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='what is encoder?', additional_kwargs={}, response_metadata={}), AIMessage(content='The encoder is a component of a transformer model. \nIt consists of multiple identical layers, each containing two sublayers: a multi-head self-attention mechanism and a feed-forward network. \nThe encoder processes the input sequence and generates a representation of its meaning.  \n', additional_kwargs={}, response_metadata={}), HumanMessage(content='how many layers it has?', additional_kwargs={}, response_metadata={}), AIMessage(content='The encoder has N = 6 layers.  \n', additional_kwargs={}, response_metadata={})])

In [22]:
# <!-- INPUT:
# ┌────────────────────────┐
# │ user input             │
# │ chat history           │
# └────────┬───────────────┘
#          ▼
# REPHRASING (LLM call 1)
# ┌────────────────────────┐
# │ rephrased question     │
# └────────┬───────────────┘
#          ▼
# RETRIEVAL (Vector search)
# ┌────────────────────────┐
# │ relevant documents     │
# └────────┬───────────────┘
#          ▼
# QA CHAIN (LLM call 2)
# ┌────────────────────────┐
# │ uses documents + chat  │
# │ generates final answer │
# └────────┬───────────────┘
#          ▼
# OUTPUT:
# "Task Decomposition is ..." -->


# <!-- 
# | Stage | What Happens                                    | Involves                              |
# | ----- | ----------------------------------------------- | ------------------------------------- |
# | **1** | User input with chat history is passed to chain | `rag_chain.invoke(...)`               |
# | **2** | Question is rephrased using history             | `create_history_aware_retriever`, LLM |
# | **3** | Rephrased question used to retrieve documents   | `ChromaRetriever`                     |
# | **4** | Final prompt built using context + chat         | `qa_prompt`                           |
# | **5** | LLM generates answer                            | Final `llm.invoke(...)`               | -->



# self made history saver qa chatbot without standard library

In [23]:
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough,RunnableMap

In [24]:
store={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [25]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

retriever_prompt=ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{user_question}")
    ]
)

In [26]:
hsitory_retrieval_chain={
    "chat_history":RunnableLambda(lambda x: get_session_history(x["session_id"]).messages),
    "user_question":RunnablePassthrough(),
    "session_id":RunnablePassthrough()
}|retriever_prompt|llm| RunnableLambda(lambda msg: msg.content)

hsitory_retrieval_chain

{
  chat_history: RunnableLambda(...),
  user_question: RunnablePassthrough(),
  session_id: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['chat_history', 'user_question'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageC

In [27]:
hsitory_retrieval_chain.invoke({"user_question":"what is encoder","session_id":"chat1"})

'What is an encoder? \n'

In [28]:
history_aware_retriever1 = hsitory_retrieval_chain | RunnableLambda(
    lambda question: retriever.get_relevant_documents(question)
)

history_aware_retriever1

{
  chat_history: RunnableLambda(...),
  user_question: RunnablePassthrough(),
  session_id: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['chat_history', 'user_question'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageC

In [29]:
history_aware_retriever1.invoke({
    "user_question": "what is encoder",
    "session_id": "abc123"
})

/var/folders/hj/8fxfbdp516z1q9b5vhdr9yq00000gn/T/ipykernel_47740/1765314843.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  lambda question: retriever.get_relevant_documents(question)


[Document(id='18a62e94-7c76-4bc6-9bc9-02549852c1a0', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'),
 Document(id='ac78eb6d-dcdd-44bb-a936-f00c96435f71', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='an

In [ ]:
system_prompt1 = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context_extracted}"
)


qa_prompt1 = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt1),
        MessagesPlaceholder("chat_history"),
        ("human", "{question}"),
    ]
)

qa_prompt1

ChatPromptTemplate(input_variables=['context_extracted', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context_extracted'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context_extracted}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [35]:
chain1={
    "context_extracted":RunnableLambda(lambda x:history_aware_retriever1.invoke({"user_question": x["question"],"session_id": x["session_id"]})),
    "chat_history":RunnableLambda(lambda x: get_session_history(x["session_id"]).messages),
    "question":RunnablePassthrough(),
    "session_id":RunnablePassthrough()
}|qa_prompt1|llm 

In [36]:
chain1.invoke({
    "question": "what is encoder",
    "session_id": "abc123"
})

AIMessage(content='The encoder is composed of a stack of N = 6 identical layers.  Each layer has two sublayers: a multi-head self-attention layer and a feed-forward network.  The encoder processes the input sequence and outputs a representation of the sequence. \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 1105, 'total_tokens': 1162, 'completion_time': 0.103636364, 'prompt_time': 0.021189355, 'queue_time': 0.257701184, 'total_time': 0.124825719}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--90674b85-1819-4e6d-9e26-d9451debdeb0-0', usage_metadata={'input_tokens': 1105, 'output_tokens': 57, 'total_tokens': 1162})

In [37]:
full_chain = RunnableLambda(lambda x: {
    "question": x["question"],
    "session_id": x["session_id"],
    "llm_output": chain1.invoke(x)  # chain1 returns an AIMessage
}) | RunnableLambda(lambda data: (
    get_session_history(data["session_id"]).add_user_message(data["question"]),
    get_session_history(data["session_id"]).add_ai_message(data["llm_output"]),
    data["llm_output"].content  # ✅ This is the actual text string to return
)[2]) 

In [38]:
full_chain.invoke({
    "question": "what is encoder",
    "session_id": "abc123"
})

'The encoder is composed of a stack of N=6 identical layers.  \nEach layer has two sub-layers: a multi-head self-attention layer and a feed-forward network.  \nThe memory keys and values come from the output of the encoder. \n'

In [39]:
store["abc123"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata={}), AIMessage(content='The encoder is composed of a stack of N = 6 identical layers. Each layer has two parts: a multi-head self-attention mechanism and a fully connected feed-forward network.  The encoder processes the input sequence and generates a representation of the input. \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 1105, 'total_tokens': 1162, 'completion_time': 0.103636364, 'prompt_time': 0.021150244, 'queue_time': 0.267926926, 'total_time': 0.124786608}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--6972ec04-b136-4dd0-8b44-cb90b8b9c355-0', usage_metadata={'input_tokens': 1105, 'output_tokens': 57, 'total_tokens': 1162}), HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata=

In [40]:
full_chain.invoke({
    "question": "how many layers it contains",
    "session_id": "abc123"
})

'The encoder contains 6 identical layers.  Each of these layers has two sub-layers. The decoder inserts an extra sub-layer, bringing the total number of layers in the decoder to slightly more than 6. \n\n\n'

In [41]:
store["abc123"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata={}), AIMessage(content='The encoder is composed of a stack of N = 6 identical layers. Each layer has two parts: a multi-head self-attention mechanism and a fully connected feed-forward network.  The encoder processes the input sequence and generates a representation of the input. \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 1105, 'total_tokens': 1162, 'completion_time': 0.103636364, 'prompt_time': 0.021150244, 'queue_time': 0.267926926, 'total_time': 0.124786608}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--6972ec04-b136-4dd0-8b44-cb90b8b9c355-0', usage_metadata={'input_tokens': 1105, 'output_tokens': 57, 'total_tokens': 1162}), HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata=

# using standard library and change variables

In [42]:
store={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [43]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

retriever_prompt=ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)

retriever_prompt

ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[l

In [44]:
# both prompt variable need to be 'chat_history1', 'input' else error

history_retriever2=create_history_aware_retriever(llm=llm,retriever=retriever,prompt=retriever_prompt)
history_retriever2

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x344cb3ed0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk'

In [45]:
system_prompt1 = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


qa_prompt1 = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt1),
        MessagesPlaceholder("history"),
        ("human", "{input}"),
    ]
)

qa_prompt1

ChatPromptTemplate(input_variables=['context', 'history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[

In [46]:
stuff_chain=create_stuff_documents_chain(llm,qa_prompt1)
stuff_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_c

In [47]:
chain2=create_retrieval_chain(history_retriever2,stuff_chain)
chain2


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x344cb3ed0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated

In [48]:
cahin_final=RunnableWithMessageHistory(
    chain2,
    get_session_history,
    input_messages_key="input",
    output_messages_key="answer",
    history_messages_key="history"
)

In [49]:
cahin_final

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x34d84a200>, input_messages_key='input', output_messages_key='answer', history_messages_key='history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [50]:
cahin_final.invoke({
    "input": "what is encoder"
},
config={"configurable":{"session_id":"abc123"}})

{'input': 'what is encoder',
 'history': [],
 'context': [Document(id='18a62e94-7c76-4bc6-9bc9-02549852c1a0', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 4, 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='encoder.'),
  Document(id='678b4e0b-1bda-4f42-bc10-46f1dfef2628', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 2, 'page_label': '3', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '',

In [51]:
store["abc123"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata={}), AIMessage(content='The encoder is a component in a transformer model. \n\nIt processes an input sequence of symbol representations (like words) and transforms them into a sequence of output representations. \n\nThese output representations capture the context and meaning of the input sequence. \n\n\n', additional_kwargs={}, response_metadata={})])

In [52]:
cahin_final.invoke({
    "input": "how many layers it contains",
}
,
config={"configurable":{"session_id":"abc123"}})

{'input': 'how many layers it contains',
 'history': [HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The encoder is a component in a transformer model. \n\nIt processes an input sequence of symbol representations (like words) and transforms them into a sequence of output representations. \n\nThese output representations capture the context and meaning of the input sequence. \n\n\n', additional_kwargs={}, response_metadata={})],
 'context': [Document(id='ce823adb-0962-4cf8-9fbf-4ca1de8ce683', metadata={'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'page': 5, 'page_label': '6', 'producer': 'pdfTeX-1.40.25', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './attention.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': '/False'}, page_content='diff

In [53]:
store["abc123"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='what is encoder', additional_kwargs={}, response_metadata={}), AIMessage(content='The encoder is a component in a transformer model. \n\nIt processes an input sequence of symbol representations (like words) and transforms them into a sequence of output representations. \n\nThese output representations capture the context and meaning of the input sequence. \n\n\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='how many layers it contains', additional_kwargs={}, response_metadata={}), AIMessage(content='The encoder contains N = 6 identical layers.  \n', additional_kwargs={}, response_metadata={})])

# sample example but without history aware retrieval

In [54]:
store={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [55]:
prompt = ChatPromptTemplate.from_messages([
                ("system", "You're an assistant who's good at {ability}"),
                MessagesPlaceholder(variable_name="history"),
                ("human", "{question}"),
            ])

chain = prompt | llm
chain

ChatPromptTemplate(input_variables=['ability', 'history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotat

In [56]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    # Uses the get_by_session_id function defined in the example
    # above.
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

chain_with_history

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x3a45df920>, input_messages_key='question', history_messages_key='history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [57]:
chain_with_history.invoke(  # noqa: T201
    {"ability": "math", "question": "What does cosine mean?"},
    config={"configurable": {"session_id": "foo"}}
)

AIMessage(content='I\'m happy to explain cosine! \n\nCosine (often abbreviated as "cos") is a trigonometric function. It relates an angle of a right triangle to the ratio of two of its sides. \n\n**Here\'s the breakdown:**\n\n* **Right Triangle:**  A triangle with one angle measuring 90 degrees.\n* **Angle:** The corner of the triangle you\'re interested in.\n* **Hypotenuse:** The longest side of the right triangle, always opposite the right angle.\n* **Adjacent Side:** The side next to the angle you\'re considering (not the hypotenuse).\n\n**The Definition:**\n\nCosine of an angle (θ) in a right triangle is defined as the ratio of the length of the adjacent side to the length of the hypotenuse:\n\n **cos(θ) = Adjacent Side / Hypotenuse**\n\n**Let me know if you\'d like:**\n\n* A visual diagram to illustrate this\n* Examples of how to use cosine in calculations\n* Information about other trigonometric functions (sine, tangent, etc.)\n', additional_kwargs={}, response_metadata={'token_u

In [58]:
store['foo']

InMemoryChatMessageHistory(messages=[HumanMessage(content='What does cosine mean?', additional_kwargs={}, response_metadata={}), AIMessage(content='I\'m happy to explain cosine! \n\nCosine (often abbreviated as "cos") is a trigonometric function. It relates an angle of a right triangle to the ratio of two of its sides. \n\n**Here\'s the breakdown:**\n\n* **Right Triangle:**  A triangle with one angle measuring 90 degrees.\n* **Angle:** The corner of the triangle you\'re interested in.\n* **Hypotenuse:** The longest side of the right triangle, always opposite the right angle.\n* **Adjacent Side:** The side next to the angle you\'re considering (not the hypotenuse).\n\n**The Definition:**\n\nCosine of an angle (θ) in a right triangle is defined as the ratio of the length of the adjacent side to the length of the hypotenuse:\n\n **cos(θ) = Adjacent Side / Hypotenuse**\n\n**Let me know if you\'d like:**\n\n* A visual diagram to illustrate this\n* Examples of how to use cosine in calculatio

In [59]:
chain_with_history.invoke(  # noqa: T201
    {"ability": "math", "question": "What's its inverse"},
    config={"configurable": {"session_id": "foo"}}
)

AIMessage(content="The inverse of cosine is called **arccosine**, often written as **cos⁻¹(x)** or **acos(x)**.\n\n**What does it do?**\n\nWhile cosine tells you the ratio of sides given an angle, arccosine does the opposite: it tells you the angle given the ratio of sides.  \n\n**Here's how it works:**\n\nIf you know  cos(θ) = x, then arccos(x) = θ.\n\n**Important points:**\n\n* **Range:** The arccosine function only gives you angles within a specific range, typically from 0 to π radians (or 0 to 180 degrees).\n* **Input:** The input to arccosine must be a number between -1 and 1 (inclusive), because the cosine function only outputs values within that range.\n\n\nLet me know if you'd like to see some examples of how arccosine is used!\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 194, 'prompt_tokens': 265, 'total_tokens': 459, 'completion_time': 0.352727273, 'prompt_time': 0.005793246, 'queue_time': 0.261208293, 'total_time': 0.358520519}, 'model_na

In [60]:
store['foo']

InMemoryChatMessageHistory(messages=[HumanMessage(content='What does cosine mean?', additional_kwargs={}, response_metadata={}), AIMessage(content='I\'m happy to explain cosine! \n\nCosine (often abbreviated as "cos") is a trigonometric function. It relates an angle of a right triangle to the ratio of two of its sides. \n\n**Here\'s the breakdown:**\n\n* **Right Triangle:**  A triangle with one angle measuring 90 degrees.\n* **Angle:** The corner of the triangle you\'re interested in.\n* **Hypotenuse:** The longest side of the right triangle, always opposite the right angle.\n* **Adjacent Side:** The side next to the angle you\'re considering (not the hypotenuse).\n\n**The Definition:**\n\nCosine of an angle (θ) in a right triangle is defined as the ratio of the length of the adjacent side to the length of the hypotenuse:\n\n **cos(θ) = Adjacent Side / Hypotenuse**\n\n**Let me know if you\'d like:**\n\n* A visual diagram to illustrate this\n* Examples of how to use cosine in calculatio